In [1]:
import tidy3d as td
import numpy as np
import pandas as pd
import tidy3d.web as web
print(td.__version__)

import matplotlib.pyplot as plt
# %matplotlib widget
from matplotlib.colors import LogNorm

# import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

import gc
import os

import tidy3d.web as web
from getpass import getpass
api_key = getpass("Enter your API key: ")
# web.configure("_blank_")
web.configure(api_key)

2.10.2


Enter your API key:  ········


Configuration saved successfully.


In [2]:
web.test()

01:16:30 Malay Peninsula Standard Time Authentication configured successfully!

# Definitions

In [3]:
h_planck = 6.62607015e-34
e_charge = 1.602176634e-19
def hz_to_ev(f): return (h_planck * f) / e_charge
def ev_to_hz(E): return (E * e_charge) / h_planck

In [4]:
def extract_monitor_data(filepath, monitor_name):
    
    # Load full simulation (unavoidable)
    sim_data = td.SimulationData.from_file(filepath)
    
    # Extract ONLY the monitor we need
    monitor_data = sim_data[monitor_name]
    
    # Delete the full simulation data immediately
    del sim_data
    gc.collect()
    
    return monitor_data

In [5]:
def get_plane_config(plane):
    """Return slicing info, labels, and field pairs for a given plane."""
    configs = {
        # ============ CHANGED: H-field plane configurations ============
        'Hxy': {'slicer': {'z': 0},
            'U_field': 'Hx', 'V_field': 'Hy',
            'xlabel': 'x (nm)', 'ylabel': 'y (nm)',
            'coord_keys': ('x', 'y')},
        
        'Hxz': {'slicer': {'y': 0},
            'U_field': 'Hx', 'V_field': 'Hz',
            'xlabel': 'x (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('x', 'z')},
        
        'Hyz': {'slicer': {'x': 0},
            'U_field': 'Hy', 'V_field': 'Hz',
            'xlabel': 'y (nm)', 'ylabel': 'z (nm)',
            'coord_keys': ('y', 'z')},
        # ============ END CHANGED ============
    }
    if plane not in configs:
        raise ValueError(f"plane must be one of {list(configs.keys())}")
    
    return configs[plane]


# ========================================


In [6]:
def get_field_component(monitor_data, monitor_data0, comp, slicer, peak, normalize):
    """Extract field component from monitor data"""
    data  = getattr(monitor_data , comp).isel(**slicer).interp(f=peak)
    data0 = getattr(monitor_data0, comp).isel(**slicer).interp(f=peak)
    
    if normalize:
        return (data - data0) / data0
    else:
        return data - data0


def get_coords(monitor_data, plane):
    """Extract and format coordinates for plotting"""
    coord_key1, coord_key2 = get_plane_config(plane)['coord_keys']
    
    # Extract coordinates and convert to nm
    coord1 = monitor_data.Hx.coords[coord_key1].values * 1e3
    coord2 = monitor_data.Hx.coords[coord_key2].values * 1e3
    
    # Ensure increasing order
    if coord1[0] > coord1[-1]:
        coord1 = coord1[::-1]
    if coord2[0] > coord2[-1]:
        coord2 = coord2[::-1]
    
    return coord1, coord2


# ========================================


In [7]:
def prepare_Hfield_data(Hx, Hy, Hz, coord1, coord2, plane):
    """Compute |H| and meshgrid from field components"""
    # Convert to real NumPy arrays
    Hx = np.real(np.array(Hx))
    Hy = np.real(np.array(Hy))
    Hz = np.real(np.array(Hz))
    
    # Expected shape
    expected_shape = (len(coord2), len(coord1))
    
    # Transpose if needed
    if Hx.shape != expected_shape:
        print(f"(Hx.shape={Hx.shape}, expected={expected_shape})")
        Hx, Hy, Hz = Hx.T, Hy.T, Hz.T
    
    # For xy plane, apply additional transpose
    # if plane == 'Hxy':
    #     Hx, Hy. Hz  = Hx.T, Hy.T, Hz.T
    #     print("[INFO] Applied xy-plane transpose for correct orientation")
    
    # Compute magnitude
    H = np.sqrt(np.abs(Hx)**2 + np.abs(Hy)**2 + np.abs(Hz)**2)
    
    # Build coordinate mesh
    horizontal_axis, vertical_axis = np.meshgrid(coord1, coord2)
    
    # Orientation diagnostics
    print(f"[DEBUG] {plane}-plane orientation check:")
    print(f"  horizontal_axis shape={horizontal_axis.shape}, "
          f"vertical_axis shape={vertical_axis.shape}")
    print(f"  H-field array shape={H.shape}")
    
    return H, horizontal_axis, vertical_axis, Hx, Hy, Hz

In [8]:
def plot_Hfield(H, horizontal_axis, vertical_axis, U, V, 
               xlabel, ylabel, plane, peak, name, monitor,
               fixed_scale, density, arrow, save_path=None):
    """Render H-field magnitude and streamlines"""
    plt.figure(figsize=(8, 6))
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{name} | {monitor} | {hz_to_ev(peak):.3f} eV")
    
    # Color scale
    if fixed_scale:
        cmap_data = H
        cbar_label = '|H| induced'  # CHANGED: Label for H-field
        vmin, vmax = np.percentile(H, [5, 99])
    else:
        cmap_data = np.abs(H) * 100
        cbar_label = '|H| (%)'
        vmin, vmax = 0, 1000
    
    # Draw color map
    plt.pcolor(horizontal_axis, vertical_axis, cmap_data, 
               cmap='viridis',  # CHANGED: Different colormap for H-field
               shading='auto',
               norm=LogNorm(vmin=vmin, vmax=100*vmax))  
    plt.colorbar(label=cbar_label)
    
    # Streamlines
    plt.streamplot(horizontal_axis, vertical_axis, U, V,
                   density=density,
                   linewidth=(H - H.min()) / (H.max() - H.min()) + 0.05,
                   color='white',
                   arrowstyle=arrow)
    
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, 
                    # bbox_inches='tight'
                   )
        print(f"  Saved: {os.path.basename(save_path)}")
    
    plt.close()

In [9]:
def plot_Hfield_stream(monitor_data, monitor_data0, monitor, peak, plane='Hxz', 
                      normalize=False, fixed_scale=True,
                      density=2.5, arrow='fancy, head_length=0.7',
                      save_path=None):
    """Orchestrate H-field extraction and plotting for a 2D plane"""
    cfg = get_plane_config(plane)
    
    # Extract H-field components (CHANGED: Hx, Hy, Hz instead of Ex, Ey, Ez)
    Hx = get_field_component(monitor_data, monitor_data0, 'Hx', cfg['slicer'], peak, normalize)
    Hy = get_field_component(monitor_data, monitor_data0, 'Hy', cfg['slicer'], peak, normalize)
    Hz = get_field_component(monitor_data, monitor_data0, 'Hz', cfg['slicer'], peak, normalize)
    
    # Get coordinates and prepare data
    coord1, coord2 = get_coords(monitor_data, plane)
    H, X, Y, Hx, Hy, Hz = prepare_Hfield_data(Hx, Hy, Hz, coord1, coord2, plane)
    
    # Select vector components for streamlines (CHANGED: Hx, Hy, Hz)
    field_map = {'Hx': Hx, 'Hy': Hy, 'Hz': Hz}
    U = field_map[cfg['U_field']]
    V = field_map[cfg['V_field']]
    
    # Plot
    plot_Hfield(H, X, Y, U, V, cfg['xlabel'], cfg['ylabel'], plane, peak, 
               name, monitor, fixed_scale, density, arrow, save_path=save_path)

    

In [10]:
def process_single_monitor(monitor_name, peak, plane, save_path):

    try:
        # Extract only this monitor from both files
        monitor_data = extract_monitor_data(f'{save_dir}/{name_file}.hdf5', monitor_name)
        monitor_data0 = extract_monitor_data(f'{save_dir}/{name_file}_empty.hdf5', monitor_name)
        
        # ============ CHANGED: plot_Hfield_stream  ============
        # Process and plot H-field
        plot_Hfield_stream(
            monitor_data, monitor_data0,
            monitor=monitor_name,
            peak=peak,
            plane=plane,
            normalize=False,
            save_path=save_path
        )
        # ============ END CHANGED ============
        
        # Free memory
        del monitor_data, monitor_data0
        gc.collect()
        
        # print(f"✓ Completed {monitor_name}\n")
        return True

    except KeyError:
        print(f"⚠ Monitor '{monitor_name}' not found in simulation data - SKIPPING")
        print(f"{'='*60}\n")
        return False
    except Exception as e:
        print(f"❌ Error processing {monitor_name}: {e}")
        print(f"{'='*60}\n")
        return False
    

# Execution

In [13]:
# --- DICTIONARY FOR IN-PLANE VALUES ---
ev_in_data = {
    'dipolEz_Th100D100G10':  [7.99, 16.19],
    'dipolEz_Th100D100G20':  [8.23, 16.45],
    'dipolEz_Th100D100G30':  [8.23, 16.52],
    'dipolEz_Th100D100G40':  [8.17, 16.52],
    'dipolEz_Th100D100G50':  [7.80, 15.81, 3.06],
    'dipolEz_Th100D100G100': [8.32, 17.01],
    'dipolEz_Th100D100G150': [8.27, 17.11],
    'dipolEz_Th100D100G200': [7.9, 15.89],
}

# --- DICTIONARY FOR OUT-PLANE VALUES ---
ev_out_data = {
    'dipolEz_Th100D100G10':  [8.05,16.17,  11.75],
    'dipolEz_Th100D100G20':  [8.29,16.4, ],
    'dipolEz_Th100D100G30':  [8.30, 16.39],
    'dipolEz_Th100D100G40':  [8.23, 16.45],
    'dipolEz_Th100D100G50':  [8.24,17.56],
    'dipolEz_Th100D100G100': [8.43, 13.59, 16.64, 19.29],
    'dipolEz_Th100D100G150': [8.37, 4.85],
    'dipolEz_Th100D100G200': [8.26],
}

save_dir = '/Users/Howfishy/Documents/0. tidy3d/2Mar Array/'
# 5. Process in a row
# List of the G-values you want to process
g_values = [
    # 10, 20, 
    # 30, 40, 
    50, 100, 
    # 150, 200
]

for g in g_values:
    name = f'dipolEz_Th100D100G{g}'
    
    if name not in ev_in_data:      # Check if this name exists in your dictionaries before processing
        print(f"Skipping {name}: No data found in dictionary.")
        continue

    print(f"--------------- Processing: {name} ------------------------")
    name_file = f"{name}_vertical_n1_nophase_in_out"  # Setup paths
    plot_dir = os.path.join(save_dir, f"field_plots_{name_file}")
    os.makedirs(plot_dir, exist_ok=True)

    # # Process IN-PLANE
    # for EV in ev_in_data[name]:
    #     process_single_monitor(
    #         monitor_name='DFT_in_plane_slice0',
    #         peak=ev_to_hz(EV),
    #         plane='Hxy',
    #         save_path=os.path.join(plot_dir, f'{EV:.2f}eV_{name}_IN_H.png')
    #     )

    # Process OUT-PLANE
    for EV in ev_out_data[name]:
        process_single_monitor(
            monitor_name='DFT_out_plane_XZ',
            peak=ev_to_hz(EV),
            plane='Hxz',
            save_path=os.path.join(plot_dir, f'{EV:.2f}eV_OUT_H.png')
        )

--------------- Processing: dipolEz_Th100D100G50 ------------------------
(Hx.shape=(77, 278), expected=(278, 77))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(278, 77), vertical_axis shape=(278, 77)
  H-field array shape=(278, 77)
  Saved: 8.24eV_OUT_H.png
(Hx.shape=(77, 278), expected=(278, 77))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(278, 77), vertical_axis shape=(278, 77)
  H-field array shape=(278, 77)
  Saved: 17.56eV_OUT_H.png
--------------- Processing: dipolEz_Th100D100G100 ------------------------
(Hx.shape=(101, 278), expected=(278, 101))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(278, 101), vertical_axis shape=(278, 101)
  H-field array shape=(278, 101)
  Saved: 8.43eV_OUT_H.png
(Hx.shape=(101, 278), expected=(278, 101))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(278, 101), vertical_axis shape=(278, 101)
  H-field array shape=(278, 101)
  Saved: 13.59eV_OUT_H.png
(Hx.shape=(101, 278), expected=(278

In [19]:
# ========= # Visualisation # =============
sim_data        = td.SimulationData.from_file(f'{save_dir}/{name}.hdf5')

sim_data.simulation.plot_3d()
print(sim_data.simulation.mediums)
for monitor in sim_data.simulation.monitors:
    print(monitor.name, monitor.type)

# Workings

In [32]:
# save_dir = '/app/local_project/10Jan holes'
# save_dir = '/Users/Howfishy/Documents/0. tidy3d/27Feb Phase/'
# os.makedirs(save_dir, exist_ok=True)



# Singlemers
# EV_Hfield = [
    # 8.27, 16.65      #Th100D100G50
    # 8.42, 16.38, #D100 L100
    # 14.81, # 5.05, 10.14, #Th100D200G50
    # 8.85, 16.13,3.68,10.32,  # D100L100
    # 3.75, 13.29,#10.54, 15.28 # D100 L25
    # 10.19, 5.35, 20.57 # D200L100  
    # 5.153, 10.14,
    # 10.24,             # D200 L25
    # 5.47, 10.18, #5.153, 10.14  # D200 L100
    # 9.679, 14.82 # dimer D200L100
    # 8.28, 16.69   # D100 L100
    # 8.399, 16.20  # D100 L100
    # 12.68,
    # 2.51 #L100 all Hx   
# ]



# Dimers
# EV_Hfield = [
#     # 8.40, 15.48, # G100 outplane
#     # 8.39, 16.31, # G50  Ez  Both vert
#     # 8.10, 17.15, # G10
    
#     # 6.32,    # G100
#     6.37,   # G50 Hx  Both vert
#     # 6.45 ,  # G10  
#     4.13, 8.10,  # Dips
#     3.32
    
#     # 8.24, 16.73,  # G100 in plane
#     # 8.26, 16.61,  # Gap50
#     # 8.15, 16.64    # G10
# ]

# EV_Hfield = [
# In plane
# 5.30, 8.28, 16.8              # 150   3.190894365, 14.85298868, 9.680526303
# 5.27, 8.23, 16.82             # 100  3.043170514, 14.63900903, 9.291105512
# 3.86, 8.32, 17.13            # 50    2.159042236, 12.30254829, 8.777088888
# 3.78, 8.18, 13.34, 17.13  # 10  2.229249162, 9.612764384, 15.10925496, 11.30604998

# Out
# 9.25, 10.84, 18.42            # 150 6.632353266, 3.622842185, 12.44095544
# 9.27, 11.55, 18.64            # 100 6.576221675, 8.15661918, 15.02129872
# 9.33, 11.33, 16.16, 21.05     # 50  5.849506085, 7.045835761, 27.97736854, 10.75176837
# 9.14, 12.03, 17.98             # 10  7.248261845, 7.622718546, 8.400214393
# ]  


# for suffix in ["_vertical_n1", "_both_vertical", ]: # "_both_random",
#     name      = 'dipolEz_Th100D100G50' +suffix   
#     name_file = name +'_nophase_out_plane'
    # name = 'dipolEz_D100L100' + '_OUT' #+ '_25'

# name      = 'dipolEz_Th100D100G10' 
# name_file = name + '_both_vertical_withphase' + '_out_plane'
# os.makedirs('field_plots_'+ name_file, exist_ok=True)


# for EV in EV_Hfield:
#     process_single_monitor(
#             monitor_name='DFT_in_plane_slice0',
#             peak=ev_to_hz(EV),
#             plane='Hxy',
#             save_path=os.path.join('field_plots_' + name_file, f'{EV:.2f}eV_IN_H.png')
#         )



(Hx.shape=(366, 318), expected=(318, 366))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(318, 366), vertical_axis shape=(318, 366)
  H-field array shape=(318, 366)
  Saved: 9.14eV_dipolEz_Th100D100G10_OUT_H.png
(Hx.shape=(366, 318), expected=(318, 366))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(318, 366), vertical_axis shape=(318, 366)
  H-field array shape=(318, 366)
  Saved: 12.03eV_dipolEz_Th100D100G10_OUT_H.png
(Hx.shape=(366, 318), expected=(318, 366))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(318, 366), vertical_axis shape=(318, 366)
  H-field array shape=(318, 366)
  Saved: 17.98eV_dipolEz_Th100D100G10_OUT_H.png


In [ ]:
# offset
EV_Hfield_offset = [
    # 8.37, 15.99        # D100L100
    # 10.17, 15.45       # D100 L25
    10.13, 5.398,           # D200L100
    # 10.23, 3.23, 18.33            # D200 L25

]

for EV in EV_Hfield_offset:
    process_single_monitor(
            monitor_name='DFT_out_plane_XZoffset',
            peak=ev_to_hz(EV),
            plane='Exz',
            save_path=os.path.join('field_plots_' + name, f'{EV:.2f}eV_Hfield_XZoffset.png')
        )

In [17]:
# Task
# EV_Hfield = 17.56
# peak_freq = ev_to_hz(EV_Hfield)

# # Define all plotting tasks
# tasks = [
#         # ('DFT_in_plane_slice0',       'Hxy', f'{EV_Hfield:.2f}eV_Hfield_IN_0.png'),
#         # ('DFT_in_plane_slice4.5',     'Hxy', f'{EV_Hfield:.2f}eV_Hfield_IN_4.5offset.png'),
#         # ('DFT_bottom_plane_slice0.5', 'Hxy', f'{EV_Hfield:.2f}eV_Hfield_BOT_0.5.png'),
#         # ('DFT_bottom_plane_slice14',  'Hxy', f'{EV_Hfield:.2f}eV_Hfield_BOT_14offset.png'),
#         ('DFT_out_plane_XZ',          'Hxz', f'{EV_Hfield:.2f}eV_Hfield_OUT_XZ.png'),
#         ('DFT_out_plane_XZoffset',    'Hxz', f'{EV_Hfield:.2f}eV_Hfield_OUT_XZoffset.png'),
#     ]

# # Process each monitor separately
# for i, (monitor, plane, filename) in enumerate(tasks, 1):
#     print(f"[{i}/{len(tasks)}]", end=" ")
#     process_single_monitor(
#         monitor_name=monitor,
#         peak=peak_freq,
#         plane=plane,
#         save_path=os.path.join('field_plots_' + name, filename)
#     )

[1/2] (Hx.shape=(253, 203), expected=(203, 253))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 253), vertical_axis shape=(203, 253)
  H-field array shape=(203, 253)
  Saved: 17.56eV_Hfield_OUT_XZ.png
✓ Completed DFT_out_plane_XZ

[2/2] (Hx.shape=(253, 203), expected=(203, 253))
[DEBUG] Hxz-plane orientation check:
  horizontal_axis shape=(203, 253), vertical_axis shape=(203, 253)
  H-field array shape=(203, 253)
  Saved: 17.56eV_Hfield_OUT_XZoffset.png
✓ Completed DFT_out_plane_XZoffset

